# Tempo Run 2025 — Balance + (optional) Enrich

Bước này gồm 2 phần:

1. **Cân bằng label A/B/C/D** (BẮT BUỘC): script tự rotate choices. **Không tốn API**, chạy ~1s.
2. **Paraphrase + Explain** (TUỲ CHỌN, tốn API): chỉ làm khi BƯỚC 5 (evaluate) cho accuracy thấp.

**Workflow khuyến nghị**:
1. Chạy 2.1 (reorder, free) → 2.3 (merge) → BƯỚC 3 (train) → BƯỚC 5 (evaluate).
2. Nếu accuracy tệ → quay lại 2.2 (paraphrase + explain, tốn API).


In [ ]:
%cd /content/finetune_1B_MCQ_VN
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")
import os
print("DASHSCOPE_API_KEY:", "OK" if os.environ.get("DASHSCOPE_API_KEY") else "MISSING (chỉ cần nếu chạy 2.2)")


## 2.1 — Balance (reorder) — recommended, no API

Script rotate `choices` sao cho label A/B/C/D phân bố đều. Không gọi API, chạy vài giây.

Với data 4041 rows lệch (B=1638, D=144), reorder đưa về ~1010 mỗi label.
Tắt reorder bằng `--no-balance` nếu muốn giữ label gốc.


In [ ]:
# 2.1: Reorder choices cho cân bằng (KHÔNG tốn API, chạy ~1s)
# → Skip 2.2 nếu accuracy đã OK sau khi train.
!python scripts/enrich_data.py \
    --in  data/processed/train.jsonl \
    --out data/processed/enriched.jsonl \
    --no-paraphrase --no-explain


## 2.2 — Paraphrase + Explain (optional, costs API)

**Chỉ chạy khi**: BƯỚC 5 (evaluate) cho accuracy thấp, cần thêm data đa dạng.

**Chi phí ước tính** (qwen3.6-max-preview, ~6s/call, 1M token free tier):
- Paraphrase 1x/row: 4041 calls × 6s ≈ 6.7h
- Explain 1x/row: thêm ~6.7h
- Tổng: ~13h cả 2 phase. Có thể chỉ chạy 1 trong 2.

Script sẽ in: model đang dùng + heartbeat progress mỗi 25 hàng (sync) hoặc 'starting/done' lines mỗi phase (async).
Đổi tần suất bằng `--progress-every N`. File `enriched.jsonl` + cache được ghi incremental mỗi checkpoint (Colab reset không mất dữ liệu).


In [ ]:
# 2.2a: (Optional) Smoke test API trước khi chạy thật — 1 call
# Bỏ comment dòng dưới nếu muốn test trước.

# from temprun.enrich import call_chat, get_client, get_model, get_model_source
# client = get_client()
# out = call_chat(client, [
#     {"role": "system", "content": "Bạn là trợ lý hữu ích."},
#     {"role": "user", "content": "Trả lời ngắn gọn: 2+2=?"},
# ], max_tokens=20, temperature=0.0)
# print('model:', get_model(), '| source:', get_model_source())
# print('output:', out)


In [ ]:
# 2.2b: Chạy paraphrase + explain (tốn API)
# Bỏ comment dòng dưới nếu muốn chạy. Mặc định paraphrase=True, explain=True.

# !python scripts/enrich_data.py \\
#     --in  data/processed/enriched.jsonl \\
#     --out data/processed/enriched.jsonl

# Chỉ paraphrase (skip explain): thêm --no-explain
# !python scripts/enrich_data.py \\
#     --in  data/processed/enriched.jsonl \\
#     --out data/processed/enriched.jsonl \\
#     --no-explain


## 2.3 — Merge: train + eval split

Gộp enriched + base, re-split 90/10 → `final/{train,eval}.jsonl`.


In [ ]:
!python scripts/merge_enriched.py


In [ ]:
# Verify final size
!wc -l data/processed/final/*.jsonl
!head -1 data/processed/final/train.jsonl | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('label:', r['label']); print('source:', r.get('_source', 'reordered'))"


## 2.4 — Backup lên HF (khuyến nghị nếu đã dùng API)

Đẩy `enriched.jsonl` + cache lên cùng repo `${HF_DATASET_REPO}` vào sub-folder `processed/`.
Quan trọng nếu bạn đã gọi API ở 2.2 — nếu Colab reset, không phải trả tiền lại.


In [ ]:
# 2.4: Push backup lên HF (cần HF_TOKEN write scope trong .env)
# Idempotent: nếu cache đã đầy, chỉ re-flush file, không tốn thêm API.
!python scripts/enrich_data.py \
    --in  data/processed/enriched.jsonl \
    --out data/processed/enriched.jsonl \
    --no-paraphrase --no-explain --push-after


## (Tuỳ chọn) Restore enriched + cache trên session mới

Nếu bạn đã push lên HF ở session trước (qua 2.4), chạy cell dưới ở session mới **trước khi** chạy `enrich_data.py` để khôi phục `enriched.jsonl` + cache.
Sau đó `enrich_data.py` chạy idempotent (đọc cache, không tốn thêm API cho row đã xong).

Cell này tải về `data/processed/_hf_cache/processed/enriched.jsonl` rồi `shutil.move` về `data/processed/enriched.jsonl`.


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")
import os

REPO = os.environ["HF_DATASET_REPO"]
TOKEN = os.environ["HF_TOKEN"]
target = repo_root() / "data" / "processed"
for fname in ("enriched.jsonl", "enriched.jsonl.cache.json"):
    in_repo = f"processed/{fname}"
    downloaded = Path(hf_hub_download(
        repo_id=REPO, filename=in_repo, repo_type="dataset", token=TOKEN,
        local_dir=target / "_hf_cache",
    ))
    final = target / fname
    if final.exists():
        final.unlink()
    shutil.move(str(downloaded), str(final))
    print(f"restored: {final}")
